# Predictive Modelling of Road Traffic Crashes in Nigeria Using Machine Learning (2021–2025)
**Reproducible pipeline for the MSc thesis of Tukur Hamid.** Raw NBS/FRSC quarterly files → feature engineering → frequency (regression) and severity (classification) models → baselines, bootstrap uncertainty, permutation importance and SHAP → prospective check on 2026-Q1 → projections → figures for Chapter 4.

**Design.** Panel of 37 states (36 states + FCT) × quarterly data. Training 2021–2024, test 2025, 2026-Q1 held out as a prospective check never used for training, tuning or thresholds. 2020-Q4 is used only as lag history. Section 5 adds the checks requested at examination: state-wise ARIMA and exponential-smoothing benchmarks, regression residual diagnostics, the Matthews correlation coefficient, and a sensitivity analysis of the severity-target definition. The 2026-Q1 check is a prospective temporal hold-out on the same data source, not external validation.

**Inputs** (place in the folder set by `DATA_DIR`):
1. `nbs-yearly-crash-2020-Q4-2021-Q1.csv`
2. `byMe_colab_crash_regression_ready.csv` (cleaned 2021-Q2 to 2023-Q4 counts)
3. `byMe_colab_crash_classification_ready.csv` (cleaned 2021–2023 with causative-factor proportions)
4. `nbs-yearly-crash-2024-2026.csv`
5. `nbs-yearly-crash-causative-factors-2024-2026.csv`
6. `nbs-yearly-crash-sex-distr-killed.csv`, `nbs-yearly-crash-sex-distr-injured.csv`, `nbs-yearly-crash-vehicle-category.csv`, `nbs-yearly-crash-vehicle-ditribution.csv` (national quarterly 2021–2025, used only for Figures 3.4–3.7)

Run all cells top to bottom (about 15–20 minutes; the randomised searches and SHAP dominate). All results are written to `OUT_DIR`. Seeds are fixed at 42.

In [1]:
# Configuration -------------------------------------------------------------
import os, sys, subprocess
DATA_DIR  = os.environ.get('DATA_DIR', 'data/')        # folder holding the input files
OUT_DIR   = os.environ.get('OUT_DIR',  'outputs')      # tables, predictions and figures are written here
TRAIN_END = int(os.environ.get('TRAIN_END', 2024))     # last training year (also used for severity thresholds)
TEST_YEAR = int(os.environ.get('TEST_YEAR', 2025))     # test year
os.makedirs(OUT_DIR, exist_ok=True)
os.environ.update(DATA_DIR=os.path.join(DATA_DIR, ''), OUT_DIR=OUT_DIR, TRAIN_END=str(TRAIN_END), TEST_YEAR=str(TEST_YEAR))
try:
    import shap
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'shap'], check=True); import shap
print('train <=', TRAIN_END, '| test =', TEST_YEAR, '| data:', DATA_DIR, '| out:', OUT_DIR)

/usr/local/lib/python3.13/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train <= 2024 | test = 2025 | data: /tmp/claude-0/nbdata/ | out: /tmp/claude-0/nbout2


# 1. Data assembly and feature engineering
Merges the NBS/FRSC quarterly crash tables (2020-Q4 to 2026-Q1) and builds the regression and classification panels.

In [2]:
import os, json
import numpy as np, pandas as pd
DATA_DIR = os.environ.get('DATA_DIR', 'data/')
OUT_DIR  = os.environ.get('OUT_DIR', '.')
TRAIN_END = int(os.environ.get('TRAIN_END', 2024)); TEST_YEAR = int(os.environ.get('TEST_YEAR', 2025))
FACTORS = ['SPV','UPD','TBT','MDV','BFL','OVL','DOT','WOT','DGD','BRD','RTV','OBS','SOS','DAD','PWR','FTQ','SLV','OTHERS']
KEY = ['YEAR','QUARTER_NUM','STATE']

In [3]:
def qnum(d):
    d = d.copy(); d['QUARTER_NUM'] = d['QUARTER'].astype(str).str.replace('Q','').astype(int); return d

old_reg  = pd.read_csv(DATA_DIR+'byMe_colab_crash_regression_ready.csv')          # 2021-Q2..2023-Q4 (earlier cleaned file)
old_cls  = pd.read_csv(DATA_DIR+'byMe_colab_crash_classification_ready.csv')      # 2021-Q1..2023-Q4, with factor proportions
new_cr   = qnum(pd.read_csv(DATA_DIR+'nbs-yearly-crash-2024-2026.csv'))           # 2024-Q1..2026-Q1
new_cf   = qnum(pd.read_csv(DATA_DIR+'nbs-yearly-crash-causative-factors-2024-2026.csv'))
back     = qnum(pd.read_csv(DATA_DIR+'nbs-yearly-crash-2020-Q4-2021-Q1.csv'))     # 2020-Q4, 2021-Q1

cols = KEY+['TOTAL CASES','NUMBER KILLED']
raw = (pd.concat([back[cols], old_reg[cols], new_cr[cols]])
         .drop_duplicates(KEY).sort_values(['STATE','YEAR','QUARTER_NUM']).reset_index(drop=True))
assert raw.groupby('STATE').size().nunique()==1 and raw['STATE'].nunique()==37
print('raw panel', raw.shape, raw.YEAR.min(), raw.YEAR.max())
raw.to_csv(OUT_DIR+'/raw_full_2020Q4_2026Q1.csv', index=False)

raw panel (814, 5) 2020 2026


## Regression features (strictly backward-looking)

In [4]:
def add_features(r):
    r = r.sort_values(['STATE','YEAR','QUARTER_NUM']).copy()
    r['TIME_INDEX'] = (r['YEAR']-2021)*4 + r['QUARTER_NUM']
    g = r.groupby('STATE')['TOTAL CASES']
    r['LAG_1'] = g.shift(1); r['LAG_2'] = g.shift(2); r['LAG_4'] = g.shift(4)
    r['ROLLING_MEAN_2'] = g.transform(lambda s: s.shift(1).rolling(2).mean())
    r['ROLLING_MEAN_4'] = g.transform(lambda s: s.shift(1).rolling(4).mean())
    lagc = ['LAG_1','LAG_2','LAG_4','ROLLING_MEAN_2','ROLLING_MEAN_4']
    r['_nan_rows'] = r[lagc].isna().any(axis=1)
    r[lagc] = r.groupby('STATE')[lagc].bfill()      # documented exception: first rows of 2021 (history starts 2020-Q4)
    r['TREND'] = r['LAG_1'] - r['LAG_2']
    r['FATALITY_RATE'] = r['NUMBER KILLED']/r['TOTAL CASES']
    return r

feat = add_features(raw)
reg = feat[(feat.YEAR>=2021)&(feat.YEAR<=TEST_YEAR)].copy()
q1_26 = feat[(feat.YEAR==2026)&(feat.QUARTER_NUM==1)].copy()
print('regression panel', reg.shape, 'NaN cells:', int(reg.drop(columns=['_nan_rows']).isna().sum().sum()))
print('rows with any back-filled lag feature:', int(reg['_nan_rows'].sum()))
print('LAG_4/ROLLING_MEAN_4 backfilled rows (2021 Q1-Q3):', int(((reg.YEAR==2021)&(reg.QUARTER_NUM<=3)).sum()))
reg.to_csv(OUT_DIR+'/regression_features_2021_2025_v2.csv', index=False)
q1_26.to_csv(OUT_DIR+'/regression_features_2026Q1.csv', index=False)

regression panel (740, 14) NaN cells: 0
rows with any back-filled lag feature: 111
LAG_4/ROLLING_MEAN_4 backfilled rows (2021 Q1-Q3): 111


## Classification panel: causative-factor proportions and tertile severity

In [5]:
prop_old = old_cls[KEY+[f+'_PROP' for f in FACTORS]]
cf = new_cf.merge(new_cr[KEY+['TOTAL CASES','NUMBER KILLED']], on=KEY)
fs = cf[FACTORS].sum(axis=1).replace(0,1)
for f in FACTORS: cf[f+'_PROP'] = cf[f]/fs
prop_new = cf[KEY+[f+'_PROP' for f in FACTORS]]
props = pd.concat([prop_old, prop_new]).drop_duplicates(KEY)
cls = reg[KEY+['TIME_INDEX','TOTAL CASES','NUMBER KILLED','FATALITY_RATE']].merge(props, on=KEY, how='left')
cls26 = q1_26[KEY+['TIME_INDEX','TOTAL CASES','NUMBER KILLED','FATALITY_RATE']].merge(props, on=KEY, how='left')
assert cls[[f+'_PROP' for f in FACTORS]].isna().sum().sum()==0 and cls26[[f+'_PROP' for f in FACTORS]].isna().sum().sum()==0
# consistency check against the earlier file's fatality rate
chk = cls.merge(old_cls[KEY+['FATALITY_RATE']], on=KEY, suffixes=('','_old'))
print('max |FR - FR_old| (2021-2023):', float((chk.FATALITY_RATE-chk.FATALITY_RATE_old).abs().max()))
# thresholds from TRAINING years only (<=2024)
tr = cls[cls.YEAR<=TRAIN_END]
lo, hi = tr.FATALITY_RATE.quantile(0.33), tr.FATALITY_RATE.quantile(0.66)
def sev(df): return pd.cut(df.FATALITY_RATE, bins=[-np.inf, lo, hi, np.inf], labels=['Low','Medium','High'])
cls['SEVERITY'] = sev(cls); cls26['SEVERITY'] = sev(cls26)
print('thresholds', lo, hi); open(OUT_DIR+'/thresholds_v2.txt','w').write(f'{lo}\n{hi}\n')
print(cls.groupby('YEAR').SEVERITY.value_counts().unstack())
cls.to_csv(OUT_DIR+'/classification_features_2021_2025_v2.csv', index=False)
cls26.to_csv(OUT_DIR+'/classification_features_2026Q1.csv', index=False)
print('classification panel', cls.shape)

max |FR - FR_old| (2021-2023): 1.1102230246251565e-16
thresholds 0.37047665056360707 0.6071628838451268
SEVERITY  Low  Medium  High
YEAR                       
2021       54      45    49
2022       50      50    48
2023       50      48    50
2024       42      52    54
2025       52      44    52


classification panel (740, 26)


# 2. Models, baselines, bootstrap, interpretability and the 2026-Q1 prospective check

In [6]:
import os, json, time, warnings
import numpy as np, pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit, StratifiedKFold, cross_val_score
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, accuracy_score, f1_score,
                             balanced_accuracy_score, cohen_kappa_score, roc_auc_score, confusion_matrix, classification_report)
from sklearn.inspection import permutation_importance
warnings.filterwarnings('ignore')
OUT_DIR = os.environ.get('OUT_DIR', '.')
SEED = 42; NB = 5000
TRAIN_END = int(os.environ.get('TRAIN_END', 2024)); TEST_YEAR = int(os.environ.get('TEST_YEAR', 2025))
FACTORS = ['SPV','UPD','TBT','MDV','BFL','OVL','DOT','WOT','DGD','BRD','RTV','OBS','SOS','DAD','PWR','FTQ','SLV','OTHERS']
PROPS = [f+'_PROP' for f in FACTORS]
R = {}   # results container, dumped to JSON

reg  = pd.read_csv(OUT_DIR+'/regression_features_2021_2025_v2.csv')
reg26= pd.read_csv(OUT_DIR+'/regression_features_2026Q1.csv')
cls  = pd.read_csv(OUT_DIR+'/classification_features_2021_2025_v2.csv')
cls26= pd.read_csv(OUT_DIR+'/classification_features_2026Q1.csv')
raw  = pd.read_csv(OUT_DIR+'/raw_full_2020Q4_2026Q1.csv')
le = LabelEncoder().fit(sorted(raw.STATE.unique()))
for d in (reg, reg26, cls, cls26): d['STATE_ENC'] = le.transform(d['STATE'])

def boot_idx(n, seed=SEED): return np.random.default_rng(seed).integers(0, n, (NB, n))
def ci(a): return [float(np.percentile(a,2.5)), float(np.percentile(a,97.5))]

## A. Crash frequency (regression)

In [7]:
FR = ['YEAR','QUARTER_NUM','TIME_INDEX','STATE_ENC','LAG_1','LAG_2','LAG_4','ROLLING_MEAN_2','ROLLING_MEAN_4','TREND']
tr = reg[reg.YEAR<=TRAIN_END].sort_values('TIME_INDEX'); te = reg[reg.YEAR==TEST_YEAR].copy()
Xtr, ytr, Xte, yte = tr[FR], tr['TOTAL CASES'], te[FR], te['TOTAL CASES'].values
R['n'] = dict(train=len(tr), test=len(te), total=len(reg))
tscv = TimeSeriesSplit(n_splits=5)

models = {'Linear Regression': LinearRegression(),
          'Random Forest': RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1),
          'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, random_state=SEED)}
def reg_metrics(y, p): return dict(MAE=float(mean_absolute_error(y,p)), RMSE=float(np.sqrt(mean_squared_error(y,p))), R2=float(r2_score(y,p)))
preds = {}; rows = []
for k,m in models.items():
    cvmae = -cross_val_score(m, Xtr, ytr, cv=tscv, scoring='neg_mean_absolute_error').mean()
    m.fit(Xtr,ytr); preds[k]=m.predict(Xte); rows.append(dict(model=k, cv_MAE=float(cvmae), **reg_metrics(yte,preds[k])))

# tuned ensembles (time-ordered CV)
rf_dist = dict(n_estimators=[100,200,300,500], max_depth=[None,5,10,15], min_samples_split=[2,5,10], min_samples_leaf=[1,2,4], max_features=['sqrt',0.5,1.0])
gb_dist = dict(n_estimators=[100,200,300,500], max_depth=[2,3,4,5], learning_rate=[0.01,0.05,0.1,0.2], min_samples_split=[2,5,10], subsample=[0.6,0.8,1.0], max_features=['sqrt','log2',None])
tuned_params = {}
for name, est, dist, it in [('Random Forest (tuned)', RandomForestRegressor(random_state=SEED, n_jobs=-1), rf_dist, 30),
                            ('Gradient Boosting (tuned)', GradientBoostingRegressor(random_state=SEED), gb_dist, 50)]:
    t=time.time()
    s = RandomizedSearchCV(est, dist, n_iter=it, cv=tscv, scoring='neg_mean_absolute_error', random_state=SEED, n_jobs=-1).fit(Xtr,ytr)
    models[name]=s.best_estimator_; preds[name]=s.best_estimator_.predict(Xte); tuned_params[name]=s.best_params_
    rows.append(dict(model=name, cv_MAE=float(-s.best_score_), **reg_metrics(yte,preds[name])))
    print(name, 'tuned in %.0fs'%(time.time()-t), s.best_params_)
R['reg_tuned_params'] = {k:{a:(None if b is None else (float(b) if isinstance(b,(float,np.floating)) else (int(b) if isinstance(b,(int,np.integer)) else b))) for a,b in v.items()} for k,v in tuned_params.items()}

# baselines (same test set)
smean = tr.groupby('STATE')['TOTAL CASES'].mean()
base = {'Naive persistence (LAG_1)': te['LAG_1'].values,
        'Seasonal naive (LAG_4)': te['LAG_4'].values,
        'State historical mean': te['STATE'].map(smean).values}
for k,p in base.items():
    preds[k]=p; rows.append(dict(model=k, cv_MAE=np.nan, **reg_metrics(yte,p)))
regtab = pd.DataFrame(rows); print(regtab.round(3).to_string())
regtab.to_csv(OUT_DIR+'/tab_reg_models.csv', index=False)

# bootstrap CIs on test observations, paired differences vs naive baseline
bi = boot_idx(len(yte))
bt = []
for k,p in preds.items():
    e = np.abs(yte-p); bm = e[bi].mean(1)
    r2b = 1 - ((yte[bi]-p[bi])**2).sum(1)/((yte[bi]-yte[bi].mean(1,keepdims=True))**2).sum(1)
    bt.append(dict(model=k, MAE=float(e.mean()), MAE_lo=ci(bm)[0], MAE_hi=ci(bm)[1], R2_lo=ci(r2b)[0], R2_hi=ci(r2b)[1]))
bt = pd.DataFrame(bt); print(bt.round(2).to_string()); bt.to_csv(OUT_DIR+'/tab_reg_bootstrap.csv', index=False)
pair = []
for ref in ['Naive persistence (LAG_1)','State historical mean']:
    for k in models:
        d = (np.abs(yte-preds[k]) - np.abs(yte-preds[ref]))[bi].mean(1)
        pair.append(dict(model=k, vs=ref, diff=float(d.mean()), lo=ci(d)[0], hi=ci(d)[1], P_better=float((d<0).mean())))
pair = pd.DataFrame(pair); print(pair.round(3).to_string()); pair.to_csv(OUT_DIR+'/tab_reg_pairwise.csv', index=False)

# selected model = lowest test MAE among the five learned models (selection on test set, as disclosed in Section 3.6.3)
sel_r = regtab[regtab.model.isin(models)].sort_values('MAE').iloc[0]['model']; R['reg_selected']=sel_r
print('SELECTED REG:', sel_r)
psel = preds[sel_r]; resid = yte-psel; sigma = float(resid.std(ddof=1)); R['reg_sigma_2025']=sigma
pd.DataFrame(dict(YEAR=te.YEAR.values, QUARTER_NUM=te.QUARTER_NUM.values, STATE=te.STATE.values, actual=yte, **{k:v for k,v in preds.items()})).to_csv(OUT_DIR+'/pred_reg_2025.csv', index=False)

# performance by quarter and by state-size band (diagnostics)
dq = pd.DataFrame(dict(Q=te.QUARTER_NUM.values, ae=np.abs(resid), ae_naive=np.abs(yte-base['Naive persistence (LAG_1)']), res=resid, y=yte))
R['reg_by_quarter'] = dq.groupby('Q').agg(MAE=('ae','mean'), MAE_naive=('ae_naive','mean'), bias=('res','mean'), mean_y=('y','mean')).round(2).to_dict('index')
# linear-model coefficients (standardised) and permutation importance of the selected model
lr = models['Linear Regression']; sc = StandardScaler().fit(Xtr)
coef = pd.DataFrame(dict(feature=FR, coef=lr.coef_, std_coef=lr.coef_*Xtr.std().values)); coef.to_csv(OUT_DIR+'/tab_reg_lr_coef.csv', index=False)
pi = permutation_importance(models[sel_r], Xte, yte, n_repeats=30, random_state=SEED, scoring='neg_mean_absolute_error')
pir = pd.DataFrame(dict(feature=FR, imp=pi.importances_mean, sd=pi.importances_std)).sort_values('imp', ascending=False); pir.to_csv(OUT_DIR+'/tab_reg_perm_importance.csv', index=False)
print(pir.round(2).to_string()); print(coef.round(3).to_string())
# importance of the two ensemble models for comparison
for nm in ['Gradient Boosting (tuned)','Random Forest (tuned)']:
    p2 = permutation_importance(models[nm], Xte, yte, n_repeats=30, random_state=SEED, scoring='neg_mean_absolute_error')
    pd.DataFrame(dict(feature=FR, imp=p2.importances_mean)).sort_values('imp',ascending=False).to_csv(OUT_DIR+f'/tab_reg_perm_importance_{nm.split()[0]}.csv', index=False)

Random Forest (tuned) tuned in 58s {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 0.5, 'max_depth': 5}


Gradient Boosting (tuned) tuned in 37s {'subsample': 1.0, 'n_estimators': 500, 'min_samples_split': 5, 'max_features': 'sqrt', 'max_depth': 2, 'learning_rate': 0.01}
                       model  cv_MAE     MAE    RMSE     R2
0          Linear Regression  16.734  16.405  21.515  0.862
1              Random Forest  17.552  14.251  20.949  0.869
2          Gradient Boosting  18.177  13.748  19.936  0.881
3      Random Forest (tuned)  16.577  14.230  20.296  0.877
4  Gradient Boosting (tuned)  16.406  14.890  21.185  0.866
5  Naive persistence (LAG_1)     NaN  14.757  19.982  0.881
6     Seasonal naive (LAG_4)     NaN  17.284  26.389  0.792
7      State historical mean     NaN  17.520  30.640  0.720


                       model    MAE  MAE_lo  MAE_hi  R2_lo  R2_hi
0          Linear Regression  16.40   14.31   18.76   0.81   0.90
1              Random Forest  14.25   12.04   16.98   0.83   0.91
2          Gradient Boosting  13.75   11.58   16.19   0.84   0.91
3      Random Forest (tuned)  14.23   12.10   16.80   0.83   0.91
4  Gradient Boosting (tuned)  14.89   12.66   17.55   0.82   0.90
5  Naive persistence (LAG_1)  14.76   12.68   17.07   0.83   0.91
6     Seasonal naive (LAG_4)  17.28   14.28   20.72   0.69   0.86
7      State historical mean  17.52   13.78   21.71   0.51   0.86
                       model                         vs   diff     lo     hi  P_better
0          Linear Regression  Naive persistence (LAG_1)  1.654 -0.192  3.350     0.035
1              Random Forest  Naive persistence (LAG_1) -0.483 -2.640  1.645     0.671
2          Gradient Boosting  Naive persistence (LAG_1) -0.994 -2.801  0.802     0.864
3      Random Forest (tuned)  Naive persistence (LAG_1) -0

          feature    imp    sd
8  ROLLING_MEAN_4  24.97  1.47
7  ROLLING_MEAN_2  10.11  0.77
4           LAG_1   4.19  0.57
5           LAG_2   1.28  0.30
1     QUARTER_NUM   0.67  0.32
3       STATE_ENC   0.50  0.21
6           LAG_4   0.38  0.37
9           TREND   0.07  0.18
0            YEAR   0.00  0.00
2      TIME_INDEX   0.00  0.00
          feature   coef  std_coef
0            YEAR -0.932    -1.043
1     QUARTER_NUM  3.304     3.698
2      TIME_INDEX -0.425    -1.961
3       STATE_ENC  0.056     0.599
4           LAG_1 -0.314   -23.554
5           LAG_2 -0.474   -36.007
6           LAG_4 -0.150   -11.622
7  ROLLING_MEAN_2  1.161    86.321
8  ROLLING_MEAN_4  0.727    53.848
9           TREND  0.159     3.956


## B. Prospective check: 2026-Q1 (never used for training, tuning or thresholds)

In [8]:
y26 = reg26['TOTAL CASES'].values; X26 = reg26[FR]
chk = {}
for tag, trainset in [('trained_2021_2024', tr), ('trained_2021_2025', reg.sort_values('TIME_INDEX'))]:
    Xa, ya = trainset[FR], trainset['TOTAL CASES']; sm = trainset.groupby('STATE')['TOTAL CASES'].mean()
    P = {}
    for k,m in models.items():
        from sklearn.base import clone
        mm = clone(m).fit(Xa,ya); P[k]=mm.predict(X26)
    P['Naive persistence (LAG_1)']=reg26['LAG_1'].values; P['Seasonal naive (LAG_4)']=reg26['LAG_4'].values; P['State historical mean']=reg26['STATE'].map(sm).values
    b26 = boot_idx(len(y26)); out=[]
    for k,p in P.items():
        e=np.abs(y26-p); bm=e[b26].mean(1); out.append(dict(model=k, **reg_metrics(y26,p), MAE_lo=ci(bm)[0], MAE_hi=ci(bm)[1], national_pred=float(p.sum()), national_actual=int(y26.sum())))
    chk[tag]=pd.DataFrame(out); chk[tag].to_csv(OUT_DIR+f'/tab_reg_2026Q1_{tag}.csv', index=False); print(tag); print(chk[tag].round(2).to_string())
    pd.DataFrame(dict(STATE=reg26.STATE.values, actual=y26, **P)).to_csv(OUT_DIR+f'/pred_reg_2026Q1_{tag}.csv', index=False)
    if tag=='trained_2021_2025':
        ps = P[sel_r]; R['cov_2026Q1_band95'] = float((np.abs(y26-ps)<=1.96*sigma).mean())
        R['cov_2025_band95'] = float((np.abs(resid)<=1.96*sigma).mean())
        d = (np.abs(y26-ps)-np.abs(y26-P['Naive persistence (LAG_1)']))[b26].mean(1); R['q1_26_sel_vs_naive']=dict(diff=float(d.mean()), ci=ci(d), P_better=float((d<0).mean()))

trained_2021_2024
                       model    MAE   RMSE    R2  MAE_lo  MAE_hi  national_pred  national_actual
0          Linear Regression  18.57  31.29  0.76   12.44   28.12        2103.20             2720
1              Random Forest  12.73  24.15  0.86    7.54   20.56        2510.30             2720
2          Gradient Boosting  13.43  23.66  0.86    8.34   20.68        2763.71             2720
3      Random Forest (tuned)  12.57  26.06  0.83    7.11   21.33        2484.46             2720
4  Gradient Boosting (tuned)  12.65  27.11  0.82    6.87   22.02        2439.38             2720
5  Naive persistence (LAG_1)  13.65  26.50  0.83    8.11   22.43        2731.00             2720
6     Seasonal naive (LAG_4)  17.89  25.29  0.84   12.84   24.35        2650.00             2720
7      State historical mean  20.44  37.94  0.65   11.34   31.80        2927.81             2720


trained_2021_2025
                       model    MAE   RMSE    R2  MAE_lo  MAE_hi  national_pred  national_actual
0          Linear Regression  13.71  27.03  0.82    8.44   22.44        2465.44             2720
1              Random Forest  12.91  25.33  0.84    7.61   21.15        2684.94             2720
2          Gradient Boosting  15.26  30.17  0.78    8.94   25.06        2719.07             2720
3      Random Forest (tuned)  12.25  25.06  0.85    6.99   20.49        2590.79             2720
4  Gradient Boosting (tuned)  12.68  26.45  0.83    7.22   21.45        2612.16             2720
5  Naive persistence (LAG_1)  13.65  26.50  0.83    8.11   22.43        2731.00             2720
6     Seasonal naive (LAG_4)  17.89  25.29  0.84   12.84   24.35        2650.00             2720
7      State historical mean  18.39  34.09  0.72   10.34   28.46        2864.55             2720


## C. Crash severity (classification)

In [9]:
SEVMAP = {'Low':0,'Medium':1,'High':2}; INV = {v:k for k,v in SEVMAP.items()}
FC = ['YEAR','QUARTER_NUM','TIME_INDEX','STATE_ENC'] + PROPS
for d in (cls, cls26): d['Y'] = d['SEVERITY'].map(SEVMAP)
ctr = cls[cls.YEAR<=TRAIN_END]; cte = cls[cls.YEAR==TEST_YEAR]
Xc, yc, Xct, yct = ctr[FC], ctr['Y'].values, cte[FC], cte['Y'].values
R['cls_counts'] = dict(train=ctr.Y.value_counts().sort_index().tolist(), test=cte.Y.value_counts().sort_index().tolist())
skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
cmodels = {'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, random_state=SEED)),
           'Random Forest': RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
           'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=SEED)}
def cls_metrics(y, p, proba=None):
    d = dict(Accuracy=float(accuracy_score(y,p)), MacroF1=float(f1_score(y,p,average='macro')), BalAcc=float(balanced_accuracy_score(y,p)), Kappa=float(cohen_kappa_score(y,p)))
    if proba is not None: d['AUC']=float(roc_auc_score(y, proba, multi_class='ovr', average='macro'))
    return d
cp, cpr, rows = {}, {}, []
for k,m in cmodels.items():
    cv = cross_val_score(m, Xc, yc, cv=skf, scoring='accuracy'); m.fit(Xc,yc); cp[k]=m.predict(Xct); cpr[k]=m.predict_proba(Xct)
    rows.append(dict(model=k, cv_acc=float(cv.mean()), cv_sd=float(cv.std()), **cls_metrics(yct,cp[k],cpr[k])))
ctuned = {}
for name, est, dist, it in [('Random Forest (tuned)', RandomForestClassifier(random_state=SEED, n_jobs=-1), rf_dist, 30),
                            ('Gradient Boosting (tuned)', GradientBoostingClassifier(random_state=SEED), gb_dist, 50)]:
    t=time.time(); s = RandomizedSearchCV(est, dist, n_iter=it, cv=skf, scoring='accuracy', random_state=SEED, n_jobs=-1).fit(Xc,yc)
    cmodels[name]=s.best_estimator_; cp[name]=s.best_estimator_.predict(Xct); cpr[name]=s.best_estimator_.predict_proba(Xct); ctuned[name]=s.best_params_
    rows.append(dict(model=name, cv_acc=float(s.best_score_), cv_sd=np.nan, **cls_metrics(yct,cp[name],cpr[name]))); print(name,'tuned %.0fs'%(time.time()-t), s.best_params_)
R['cls_tuned_params'] = {k:{a:(None if b is None else (float(b) if isinstance(b,(float,np.floating)) else (int(b) if isinstance(b,(int,np.integer)) else b))) for a,b in v.items()} for k,v in ctuned.items()}
# baselines
maj = int(pd.Series(yc).mode()[0]); cp['Majority class']=np.full(len(yct), maj)
sm_mode = ctr.groupby('STATE')['Y'].agg(lambda s: s.mode().iloc[0]); cp['State modal class (training years)'] = cte['STATE'].map(sm_mode).values.astype(int)
allc = cls.sort_values(['STATE','YEAR','QUARTER_NUM']).copy(); allc['Y_LAG1']=allc.groupby('STATE')['Y'].shift(1)
cp['Previous-quarter class'] = allc.loc[cte.index.intersection(allc.index)].loc[cte.index,'Y_LAG1'].values.astype(int)
for k in ['Majority class','State modal class (training years)','Previous-quarter class']:
    rows.append(dict(model=k, cv_acc=np.nan, cv_sd=np.nan, **cls_metrics(yct,cp[k])))
ctab = pd.DataFrame(rows); print(ctab.round(3).to_string()); ctab.to_csv(OUT_DIR+'/tab_cls_models.csv', index=False)
# bootstrap
bci = boot_idx(len(yct)); cb=[]
def f1m_boot(y, p, idx):
    out=np.empty(len(idx))
    for i,ix in enumerate(idx): out[i]=f1_score(y[ix],p[ix],average='macro')
    return out
for k,p in cp.items():
    acc=(p==yct); ab=acc[bci].mean(1); fb=f1m_boot(yct,p,bci)
    cb.append(dict(model=k, Acc=float(acc.mean()), Acc_lo=ci(ab)[0], Acc_hi=ci(ab)[1], F1=float(f1_score(yct,p,average='macro')), F1_lo=ci(fb)[0], F1_hi=ci(fb)[1]))
cb=pd.DataFrame(cb); print(cb.round(3).to_string()); cb.to_csv(OUT_DIR+'/tab_cls_bootstrap.csv', index=False)
cpair=[]
for ref in ['Majority class','State modal class (training years)','Previous-quarter class']:
    for k in cmodels:
        d=((cp[k]==yct).astype(float)-(cp[ref]==yct).astype(float))[bci].mean(1)
        cpair.append(dict(model=k, vs=ref, diff=float(d.mean()), lo=ci(d)[0], hi=ci(d)[1], P_better=float((d>0).mean())))
cpair=pd.DataFrame(cpair); print(cpair.round(3).to_string()); cpair.to_csv(OUT_DIR+'/tab_cls_pairwise.csv', index=False)
sel_c = ctab[ctab.model.isin(cmodels)].sort_values('Accuracy', ascending=False).iloc[0]['model']; R['cls_selected']=sel_c; print('SELECTED CLS:', sel_c)
pc = cp[sel_c]; cm = confusion_matrix(yct, pc); R['cls_confusion']=cm.tolist()
R['cls_report'] = classification_report(yct, pc, target_names=['Low','Medium','High'], output_dict=True)
pd.DataFrame(dict(YEAR=cte.YEAR.values, QUARTER_NUM=cte.QUARTER_NUM.values, STATE=cte.STATE.values, actual=yct, **{k:v for k,v in cp.items()})).to_csv(OUT_DIR+'/pred_cls_2025.csv', index=False)
# ablations with the selected hyper-parameters
abl=[]
for nm, cols in [('All 22 features', FC), ('Causative proportions only (18)', PROPS), ('Time and state only (4)', ['YEAR','QUARTER_NUM','TIME_INDEX','STATE_ENC']), ('Proportions + state (19)', PROPS+['STATE_ENC'])]:
    from sklearn.base import clone
    mm = clone(cmodels[sel_c]).fit(ctr[cols], yc); pp = mm.predict(cte[cols])
    abl.append(dict(features=nm, Accuracy=float(accuracy_score(yct,pp)), MacroF1=float(f1_score(yct,pp,average='macro'))))
abl=pd.DataFrame(abl); print(abl.round(3).to_string()); abl.to_csv(OUT_DIR+'/tab_cls_ablation.csv', index=False)
# permutation importance + SHAP
pic = permutation_importance(cmodels[sel_c], Xct, yct, n_repeats=30, random_state=SEED, scoring='accuracy')
picr = pd.DataFrame(dict(feature=FC, imp=pic.importances_mean, sd=pic.importances_std)).sort_values('imp',ascending=False); picr.to_csv(OUT_DIR+'/tab_cls_perm_importance.csv', index=False); print(picr.round(4).head(12).to_string())
import shap
bg = Xc.sample(100, random_state=SEED)
expl = shap.Explainer(cmodels[sel_c].predict_proba, bg, seed=SEED); t=time.time(); sv = expl(Xct); print('SHAP %.0fs'%(time.time()-t), sv.values.shape)
sa = np.abs(sv.values)   # (n, f, classes)
shp = pd.DataFrame(sa.mean(0), index=FC, columns=['Low','Medium','High']); shp['overall']=shp.mean(1); shp=shp.sort_values('overall',ascending=False); shp.to_csv(OUT_DIR+'/tab_cls_shap.csv'); print(shp.round(4).head(12).to_string())
np.save(OUT_DIR+'/shap_values_test.npy', sv.values)

Random Forest (tuned) tuned 81s {'n_estimators': 300, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 5}


Gradient Boosting (tuned) tuned 209s {'subsample': 0.8, 'n_estimators': 500, 'min_samples_split': 2, 'max_features': None, 'max_depth': 4, 'learning_rate': 0.01}
                                model  cv_acc  cv_sd  Accuracy  MacroF1  BalAcc  Kappa    AUC
0                 Logistic Regression   0.443  0.025     0.426    0.415   0.422  0.138  0.601
1                       Random Forest   0.488  0.042     0.432    0.430   0.436  0.156  0.647
2                   Gradient Boosting   0.498  0.048     0.453    0.451   0.456  0.185  0.606
3               Random Forest (tuned)   0.529    NaN     0.507    0.498   0.512  0.266  0.669
4           Gradient Boosting (tuned)   0.517    NaN     0.493    0.488   0.499  0.248  0.647
5                      Majority class     NaN    NaN     0.351    0.173   0.333  0.000    NaN
6  State modal class (training years)     NaN    NaN     0.574    0.555   0.561  0.357    NaN
7              Previous-quarter class     NaN    NaN     0.534    0.527   0.524  0.300

                                model    Acc  Acc_lo  Acc_hi     F1  F1_lo  F1_hi
0                 Logistic Regression  0.426   0.345   0.507  0.415  0.336  0.495
1                       Random Forest  0.432   0.351   0.514  0.430  0.349  0.509
2                   Gradient Boosting  0.453   0.372   0.534  0.451  0.366  0.532
3               Random Forest (tuned)  0.507   0.426   0.588  0.498  0.414  0.576
4           Gradient Boosting (tuned)  0.493   0.412   0.574  0.488  0.406  0.567
5                      Majority class  0.351   0.277   0.426  0.173  0.145  0.199
6  State modal class (training years)  0.574   0.493   0.649  0.555  0.475  0.628
7              Previous-quarter class  0.534   0.453   0.615  0.527  0.450  0.600
                        model                                  vs   diff     lo     hi  P_better
0         Logistic Regression                      Majority class  0.074 -0.020  0.169     0.926
1               Random Forest                      Majority class  0

                          features  Accuracy  MacroF1
0                  All 22 features     0.507    0.498
1  Causative proportions only (18)     0.446    0.437
2          Time and state only (4)     0.486    0.481
3         Proportions + state (19)     0.493    0.488


        feature     imp      sd
8      BFL_PROP  0.1133  0.0259
3     STATE_ENC  0.0624  0.0244
6      TBT_PROP  0.0608  0.0222
7      MDV_PROP  0.0297  0.0146
20     SLV_PROP  0.0282  0.0121
4      SPV_PROP  0.0178  0.0090
10     DOT_PROP  0.0083  0.0051
9      OVL_PROP  0.0070  0.0085
15     OBS_PROP  0.0059  0.0100
14     RTV_PROP  0.0056  0.0109
17     DAD_PROP  0.0047  0.0058
1   QUARTER_NUM  0.0029  0.0048


...

PermutationExplainer explainer:  99%|█████████▉| 147/148 [03:22<00:01,  1.28s/it]

PermutationExplainer explainer: 100%|██████████| 148/148 [03:23<00:00,  1.27s/it]

PermutationExplainer explainer: 149it [03:25,  1.25s/it]                         

PermutationExplainer explainer: 149it [03:25,  1.39s/it]

SHAP 205s (148, 22, 3)
                Low  Medium    High  overall
BFL_PROP     0.0147  0.0612  0.0635   0.0465
STATE_ENC    0.0369  0.0162  0.0474   0.0335
TBT_PROP     0.0383  0.0257  0.0130   0.0257
SLV_PROP     0.0187  0.0229  0.0086   0.0167
RTV_PROP     0.0095  0.0128  0.0221   0.0148
WOT_PROP     0.0209  0.0092  0.0135   0.0145
OTHERS_PROP  0.0150  0.0109  0.0052   0.0104
MDV_PROP     0.0097  0.0060  0.0068   0.0075
SPV_PROP     0.0089  0.0079  0.0042   0.0070
DGD_PROP     0.0074  0.0036  0.0094   0.0068
DAD_PROP     0.0050  0.0049  0.0077   0.0059
OVL_PROP     0.0067  0.0019  0.0076   0.0054


## D. Classification: 2026-Q1 prospective check

In [10]:
from sklearn.base import clone
y26c = cls26['Y'].values; X26c = cls26[FC]
chk_c=[]
for tag, tset in [('trained_2021_2024', ctr), ('trained_2021_2025', cls)]:
    P={}
    for k,m in cmodels.items(): P[k]=clone(m).fit(tset[FC], tset['Y'].values).predict(X26c)
    mj = int(tset['Y'].mode()[0]); P['Majority class']=np.full(len(y26c), mj)
    smm = tset.groupby('STATE')['Y'].agg(lambda s: s.mode().iloc[0]); P['State modal class']=cls26['STATE'].map(smm).values.astype(int)
    b=boot_idx(len(y26c))
    for k,p in P.items():
        a=(p==y26c); ab=a[b].mean(1); chk_c.append(dict(trained=tag, model=k, Accuracy=float(a.mean()), lo=ci(ab)[0], hi=ci(ab)[1], MacroF1=float(f1_score(y26c,p,average='macro'))))
    if tag=='trained_2021_2025':
        ps=P[sel_c]; d=((ps==y26c).astype(float)-(P['Majority class']==y26c).astype(float))[b].mean(1)
        R['q1_26_cls_sel_vs_majority']=dict(diff=float(d.mean()), ci=ci(d), P_better=float((d>0).mean()))
        R['q1_26_cls_confusion']=confusion_matrix(y26c, ps, labels=[0,1,2]).tolist(); R['q1_26_true_counts']=np.bincount(y26c, minlength=3).tolist()
chk_c=pd.DataFrame(chk_c); print(chk_c.round(3).to_string()); chk_c.to_csv(OUT_DIR+'/tab_cls_2026Q1.csv', index=False)

              trained                      model  Accuracy     lo     hi  MacroF1
0   trained_2021_2024        Logistic Regression     0.486  0.324  0.649    0.376
1   trained_2021_2024              Random Forest     0.486  0.324  0.649    0.462
2   trained_2021_2024          Gradient Boosting     0.378  0.216  0.541    0.366
3   trained_2021_2024      Random Forest (tuned)     0.486  0.324  0.649    0.490
4   trained_2021_2024  Gradient Boosting (tuned)     0.514  0.351  0.676    0.476
5   trained_2021_2024             Majority class     0.459  0.297  0.622    0.210
6   trained_2021_2024          State modal class     0.514  0.351  0.676    0.502
7   trained_2021_2025        Logistic Regression     0.459  0.297  0.622    0.394
8   trained_2021_2025              Random Forest     0.541  0.378  0.703    0.530
9   trained_2021_2025          Gradient Boosting     0.432  0.270  0.595    0.421
10  trained_2021_2025      Random Forest (tuned)     0.514  0.351  0.676    0.502
11  trained_2021

## E. Frequency-severity relationship (state level) and quarterly residual diagnostics

In [11]:
from scipy.stats import spearmanr
st = cls.groupby('STATE').agg(cases=('TOTAL CASES','mean'), killed=('NUMBER KILLED','mean'), fr=('FATALITY_RATE','mean'), mode_sev=('Y', lambda s: s.mode().iloc[0]))
st['fr_pooled'] = cls.groupby('STATE')['NUMBER KILLED'].sum()/cls.groupby('STATE')['TOTAL CASES'].sum()
rho, p = spearmanr(st.cases, st.fr_pooled); R['decoupling_state']=dict(rho=float(rho), p=float(p))
rho2, p2 = spearmanr(cls['TOTAL CASES'], cls['FATALITY_RATE']); R['decoupling_obs']=dict(rho=float(rho2), p=float(p2))
st['mode_sev']=st.mode_sev.map(INV); st.to_csv(OUT_DIR+'/tab_state_profile.csv'); print(st.sort_values('cases',ascending=False).head(8).round(3).to_string()); print(R['decoupling_state'], R['decoupling_obs'])
# persistence of state severity: share of state-years where yearly modal class equals previous year's
ym = cls.groupby(['STATE','YEAR'])['Y'].agg(lambda s: s.mode().iloc[0]).unstack()
R['state_year_mode_persistence'] = float(np.mean([(ym[y]==ym[y-1]).mean() for y in range(2022,TEST_YEAR+1)]))

           cases  killed     fr mode_sev  fr_pooled
STATE                                              
FCT       298.50   68.55  0.234      Low      0.230
Ogun      256.25   99.25  0.396   Medium      0.387
Kaduna    185.25  146.90  0.794     High      0.793
Nasarawa  179.30   48.95  0.285      Low      0.273
Oyo       149.05   75.45  0.508   Medium      0.506
Lagos     140.60   37.55  0.267      Low      0.267
Niger     128.25  101.50  0.813     High      0.791
Kogi      106.85   65.50  0.612     High      0.613
{'rho': -0.20578473210052156, 'p': 0.221743079447059} {'rho': -0.10484026652257587, 'p': 0.0043034968688021365}


## F. Forward-looking projections 2026-Q2 .. 2027-Q4 (deployment models refit on all data through 2026-Q1)

In [12]:
full_reg = pd.concat([reg, reg26]).sort_values('TIME_INDEX')
dep = clone(models[sel_r]).fit(full_reg[FR], full_reg['TOTAL CASES'])
series = {s: g.sort_values(['YEAR','QUARTER_NUM'])['TOTAL CASES'].tolist() for s,g in raw.groupby('STATE')}
fc=[]; steps=[(2026,2),(2026,3),(2026,4),(2027,1),(2027,2),(2027,3),(2027,4)]
for h,(y,q) in enumerate(steps, start=1):
    rows_=[]
    for s in series:
        v=series[s]; ti=(y-2021)*4+q
        rows_.append(dict(YEAR=y,QUARTER_NUM=q,TIME_INDEX=ti,STATE=s,STATE_ENC=int(le.transform([s])[0]),LAG_1=v[-1],LAG_2=v[-2],LAG_4=v[-4],ROLLING_MEAN_2=np.mean(v[-2:]),ROLLING_MEAN_4=np.mean(v[-4:]),TREND=v[-1]-v[-2]))
    Xf=pd.DataFrame(rows_); pf=np.maximum(dep.predict(Xf[FR]),0)
    for s,p_ in zip(Xf.STATE,pf): series[s].append(float(p_))
    Xf['pred']=pf; Xf['h']=h; Xf['band']=1.96*sigma*np.sqrt(h); fc.append(Xf)
fc=pd.concat(fc); fc.to_csv(OUT_DIR+'/forecast_reg_2026Q2_2027Q4.csv', index=False)
nat = fc.groupby(['YEAR','QUARTER_NUM']).agg(pred=('pred','sum'), h=('h','first'), band_indiv=('band','first'), n=('pred','size')).reset_index()
# national band under independence across states: sqrt(37)*sigma*sqrt(h) (heuristic, optimistic) 
nat['band_nat_indep']=1.96*sigma*np.sqrt(37)*np.sqrt(nat.h); nat.to_csv(OUT_DIR+'/forecast_national.csv', index=False); print(nat.round(1).to_string())
# severity projection under a 'recent causative profile' scenario (mean proportions of last 4 observed quarters)
cls_all = pd.concat([cls, cls26]).sort_values(['STATE','TIME_INDEX'])
rec = cls_all.groupby('STATE').tail(4).groupby('STATE')[PROPS].mean().rename_axis(None)
depc = clone(cmodels[sel_c]).fit(cls_all[FC], cls_all['Y'].values)
sev_rows=[]
for (y,q) in steps:
    d = rec.copy(); d['YEAR']=y; d['QUARTER_NUM']=q; d['TIME_INDEX']=(y-2021)*4+q; d['STATE']=d.index; d['STATE_ENC']=le.transform(d.index)
    d['pred']=depc.predict(d[FC]); sev_rows.append(d[['YEAR','QUARTER_NUM','STATE','pred']])
sevf=pd.concat(sev_rows); sevf['pred_label']=sevf.pred.map(INV); sevf.to_csv(OUT_DIR+'/forecast_severity_2026Q2_2027Q4.csv', index=False)
modal = sevf.groupby('STATE')['pred'].agg(lambda s: s.mode().iloc[0]).map(INV); obs25 = cls[cls.YEAR==TEST_YEAR].groupby('STATE')['Y'].agg(lambda s: s.mode().iloc[0]).map(INV)
R['sev_forecast_modal_counts']=modal.value_counts().to_dict(); R['sev_forecast_vs_2025_agree']=float((modal==obs25).mean())
comb = fc.assign(Y=fc.YEAR).groupby(['STATE']).apply(lambda g: pd.Series(dict(cases_2026_Q2_Q4=g[g.YEAR==2026].pred.sum(), cases_2027=g[g.YEAR==2027].pred.sum()))).join(modal.rename('sev_modal')).join(obs25.rename('sev_2025_modal'))
comb.to_csv(OUT_DIR+'/forecast_combined_by_state.csv'); print(comb.sort_values('cases_2027',ascending=False).head(8).round(1).to_string())
json.dump(R, open(OUT_DIR+'/results.json','w'), indent=1, default=lambda o: o.item() if hasattr(o,'item') else str(o))
print('DONE')

   YEAR  QUARTER_NUM    pred  h  band_indiv   n  band_nat_indep
0  2026            2  2723.8  1        39.0  37           237.0
1  2026            3  2470.5  2        55.1  37           335.1
2  2026            4  2807.2  3        67.5  37           410.4
3  2027            1  2733.6  4        77.9  37           473.9
4  2027            2  2708.8  5        87.1  37           529.9
5  2027            3  2521.5  6        95.4  37           580.4
6  2027            4  2820.9  7       103.1  37           626.9


          cases_2026_Q2_Q4  cases_2027 sev_modal sev_2025_modal
STATE                                                          
Kaduna              1004.2      1522.0      High           High
Ogun                 571.7       807.8    Medium         Medium
FCT                  561.4       723.8       Low            Low
Oyo                  472.3       594.8    Medium         Medium
Lagos                421.3       528.8       Low            Low
Kano                 308.6       383.5      High           High
Niger                286.2       377.5      High           High
Nasarawa             287.0       364.0       Low         Medium
DONE


# 3. Figures for Chapter 4

In [13]:
import os, json
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
OUT_DIR = os.environ.get('OUT_DIR','.'); FIG = OUT_DIR+'/figs'; os.makedirs(FIG, exist_ok=True)
R = json.load(open(OUT_DIR+'/results.json'))
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False,'figure.dpi':150,'savefig.bbox':'tight'})
BLUE, GREY, ORANGE, RED, GREEN = '#2F5597', '#8c8c8c', '#C55A11', '#C00000', '#548235'
def save(name): plt.savefig(f'{FIG}/{name}.png', dpi=200); plt.close()

# Fig 4.1 regression MAE with 95% bootstrap CI
b = pd.read_csv(OUT_DIR+'/tab_reg_bootstrap.csv').iloc[::-1].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(7.2,4.2))
cols = [GREY if ('naive' in m.lower() or 'mean' in m.lower()) else BLUE for m in b.model]
ax.errorbar(b.MAE, range(len(b)), xerr=[b.MAE-b.MAE_lo, b.MAE_hi-b.MAE], fmt='none', ecolor='#555', capsize=3, lw=1.2)
ax.scatter(b.MAE, range(len(b)), c=cols, s=45, zorder=3)
nv = float(b.loc[b.model.str.startswith('Naive'),'MAE'].iloc[0]); ax.axvline(nv, ls='--', c=GREY, lw=1); ax.text(nv+0.1, -0.75, 'naive persistence', color=GREY, fontsize=8)
ax.set_yticks(range(len(b))); ax.set_yticklabels(b.model); ax.set_xlabel('Mean absolute error, 2025 test set (crash cases per state-quarter)')
for i,r in b.iterrows(): ax.text(r.MAE_hi+0.3, i, f'{r.MAE:.1f}', va='center', fontsize=8)
ax.set_title('Frequency models and baselines: test MAE with 95% bootstrap interval', fontsize=10); save('fig4_1_reg_mae')

# Fig 4.2 predicted vs actual 2025
p = pd.read_csv(OUT_DIR+'/pred_reg_2025.csv'); sel = R['reg_selected']
fig, axs = plt.subplots(1,2, figsize=(8.4,3.9), sharex=True, sharey=True)
for ax, col, ttl in [(axs[0], sel, f'{sel}'), (axs[1], 'Naive persistence (LAG_1)', 'Naive persistence (LAG_1)')]:
    ax.scatter(p.actual, p[col], s=14, alpha=.7, c=BLUE if col==sel else GREY)
    m = max(p.actual.max(), p[col].max())*1.03; ax.plot([0,m],[0,m],'k--',lw=.8); ax.set_title(ttl, fontsize=9); ax.set_xlabel('Actual cases')
    top = p.assign(err=(p.actual-p[col]).abs()).nlargest(3,'err')
    for _,r in top.iterrows(): ax.annotate(r.STATE, (r.actual, r[col]), fontsize=7, xytext=(3,3), textcoords='offset points')
axs[0].set_ylabel('Predicted cases'); fig.suptitle('Predicted versus actual crash cases, 2025 (148 state-quarters)', fontsize=10, y=1.0); save('fig4_2_pred_actual')

# Fig 4.3 regression permutation importance
pi = pd.read_csv(OUT_DIR+'/tab_reg_perm_importance.csv').sort_values('imp')
fig, ax = plt.subplots(figsize=(6.4,3.8)); ax.barh(pi.feature, pi.imp, xerr=pi.sd, color=BLUE, ecolor='#555', capsize=2)
ax.set_xlabel('Increase in test MAE when feature is permuted (cases)'); ax.set_title(f'Permutation importance, {sel}', fontsize=10); save('fig4_3_reg_perm')

# Fig 4.4 national forecast
raw = pd.read_csv(OUT_DIR+'/raw_full_2020Q4_2026Q1.csv'); h = raw[raw.YEAR>=2021].groupby(['YEAR','QUARTER_NUM'])['TOTAL CASES'].sum().reset_index()
h['lab'] = h.YEAR.astype(str)+'-Q'+h.QUARTER_NUM.astype(str); nat = pd.read_csv(OUT_DIR+'/forecast_national.csv'); nat['lab']=nat.YEAR.astype(str)+'-Q'+nat.QUARTER_NUM.astype(str)
fig, ax = plt.subplots(figsize=(8.2,3.9)); x1=np.arange(len(h)); x2=np.arange(len(h), len(h)+len(nat))
ax.plot(x1, h['TOTAL CASES'], c=BLUE, marker='o', ms=3, label='Observed (NBS/FRSC)')
ax.plot(np.r_[x1[-1],x2], np.r_[h['TOTAL CASES'].iloc[-1], nat.pred], c=ORANGE, marker='o', ms=3, ls='--', label='Projected')
ax.fill_between(x2, nat.pred-nat.band_nat_indep, nat.pred+nat.band_nat_indep, color=ORANGE, alpha=.15, label='Approximate 95% band (heuristic)')
labs = list(h.lab)+list(nat.lab); ax.set_xticks(range(0,len(labs),2)); ax.set_xticklabels(labs[::2], rotation=60, ha='right', fontsize=7)
ax.set_ylabel('National total crash cases per quarter'); ax.legend(frameon=False, fontsize=8, loc='lower left'); ax.set_ylim(0, None)
ax.set_title('Observed national totals (2021-Q1 to 2026-Q1) and projections to 2027-Q4', fontsize=10); save('fig4_4_forecast')

# Fig 4.5 2026-Q1 prospective check (frequency)
q = pd.read_csv(OUT_DIR+'/pred_reg_2026Q1_trained_2021_2025.csv')
fig, axs = plt.subplots(1,2, figsize=(8.4,3.9), sharex=True, sharey=True)
for ax, col, c in [(axs[0], sel, BLUE), (axs[1], 'Naive persistence (LAG_1)', GREY)]:
    ax.scatter(q.actual, q[col], s=18, c=c); m=max(q.actual.max(), q[col].max())*1.03; ax.plot([0,m],[0,m],'k--',lw=.8); ax.set_title(col, fontsize=9); ax.set_xlabel('Actual 2026-Q1 cases')
    for _,r in q.assign(err=(q.actual-q[col]).abs()).nlargest(3,'err').iterrows(): ax.annotate(r.STATE,(r.actual,r[col]),fontsize=7,xytext=(3,3),textcoords='offset points')
axs[0].set_ylabel('Predicted cases'); fig.suptitle('Prospective check: predicted versus actual cases, 2026-Q1 (37 states)', fontsize=10, y=1.0); save('fig4_5_q1_2026')

# Fig 4.6 classification accuracy with CI
c = pd.read_csv(OUT_DIR+'/tab_cls_bootstrap.csv').iloc[::-1].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(7.2,4.2)); cols=[GREY if m in ('Majority class','State modal class (training years)','Previous-quarter class') else BLUE for m in c.model]
ax.errorbar(c.Acc, range(len(c)), xerr=[c.Acc-c.Acc_lo, c.Acc_hi-c.Acc], fmt='none', ecolor='#555', capsize=3); ax.scatter(c.Acc, range(len(c)), c=cols, s=45, zorder=3)
ax.axvline(1/3, ls=':', c=GREY, lw=1); ax.set_yticks(range(len(c))); ax.set_yticklabels(c.model); ax.set_xlabel('Accuracy, 2025 test set')
for i,r in c.iterrows(): ax.text(r.Acc_hi+.008, i, f'{100*r.Acc:.1f}%', va='center', fontsize=8)
ax.set_title('Severity classifiers and baselines: test accuracy with 95% bootstrap interval', fontsize=10); save('fig4_6_cls_acc')

# Fig 4.7 confusion matrix
cm = np.array(R['cls_confusion']); fig, ax = plt.subplots(figsize=(4.4,3.8)); ax.imshow(cm, cmap='Blues')
for i in range(3):
    for j in range(3): ax.text(j,i,cm[i,j],ha='center',va='center',color='white' if cm[i,j]>cm.max()/2 else 'black',fontsize=12)
ax.set_xticks(range(3)); ax.set_yticks(range(3)); ax.set_xticklabels(['Low','Medium','High']); ax.set_yticklabels(['Low','Medium','High']); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f"{R['cls_selected']}\n(2025 test set)", fontsize=9); ax.spines[:].set_visible(False); save('fig4_7_confusion')

# Fig 4.8 SHAP by class
s = pd.read_csv(OUT_DIR+'/tab_cls_shap.csv', index_col=0).head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(6.6,4.2)); left=np.zeros(len(s))
for cl,cc in [('Low',GREEN),('Medium','#BF9000'),('High',RED)]:
    ax.barh(s.index, s[cl], left=left, color=cc, label=cl); left+=s[cl].values
ax.set_xlabel('Mean |SHAP value| (change in predicted class probability)'); ax.legend(title='Severity class', frameon=False, fontsize=8); ax.set_title(f"SHAP importance, {R['cls_selected']} (2025 test set)", fontsize=10); save('fig4_8_shap')

# Fig 4.9 frequency vs severity
st = pd.read_csv(OUT_DIR+'/tab_state_profile.csv'); colmap={'Low':GREEN,'Medium':'#BF9000','High':RED}
fig, ax = plt.subplots(figsize=(7.0,4.4))
for k,g in st.groupby('mode_sev'): ax.scatter(g.cases, g.fr_pooled, c=colmap[k], s=34, label=f'{k} (modal class)', alpha=.85)
ax.set_xscale('log'); ax.set_xlabel('Mean crash cases per quarter, 2021-2025 (log scale)'); ax.set_ylabel('Fatality rate (killed / cases), pooled')
for s_ in ['FCT','Kaduna','Kano','Ogun','Lagos','Nasarawa','Bauchi','Rivers']:
    r_ = st[st.STATE==s_]
    if len(r_): ax.annotate(s_, (r_.cases.iloc[0], r_.fr_pooled.iloc[0]), fontsize=8, xytext=(4,4), textcoords='offset points')
ax.legend(frameon=False, fontsize=8); ax.set_title(f"Crash frequency versus fatality rate by state (Spearman rho = {R['decoupling_state']['rho']:.2f})", fontsize=10); save('fig4_9_decoupling')
print('figures written'); print(os.listdir(FIG))

figures written
['fig4_1_reg_mae.png', 'fig4_9_decoupling.png', 'fig4_3_reg_perm.png', 'fig4_2_pred_actual.png', 'fig4_6_cls_acc.png', 'fig4_4_forecast.png', 'fig4_8_shap.png', 'fig4_5_q1_2026.png', 'fig4_7_confusion.png']


## Section 4. Descriptive figures 3.4–3.7 (casualties by sex, vehicles by category and type)
The four NBS tables below are national quarterly totals for 2021-Q1 to 2025-Q4. They carry no state identifier, so they are descriptive only and are not model features.

In [14]:
import pandas as pd, glob, json, os, matplotlib
import matplotlib.pyplot as plt, seaborn as sns
DD=os.environ.get('DATA_DIR','data/'); OD=os.environ.get('OUT_DIR','.'); os.makedirs(os.path.join(OD,'figs'),exist_ok=True)
def load_nbs(pattern):
    d=pd.read_csv(glob.glob(os.path.join(DD,f'*{pattern}*.csv'))[0],dtype=str); d.columns=[c.strip() for c in d.columns]
    d=d.rename(columns={d.columns[2]:'CAT',d.columns[3]:'N'})      # the 3rd/4th columns are mislabelled in some files
    d['CAT']=d['CAT'].str.strip(); d['N']=d['N'].str.replace(',','').str.strip().astype(float); return d
spec=[('sex-distr-killed','fig3_4_killed_sex','Distribution of Persons Killed by Sex (2021-2025)','Sex','Total Persons Killed',(8,5),0),
      ('sex-distr-injured','fig3_5_injured_sex','Distribution of Persons Injured by Sex (2021-2025)','Sex','Total Persons Injured',(8,5),0),
      ('vehicle-category','fig3_6_veh_category','Distribution of Vehicle Categories Involved in Road Traffic Crashes (2021-2025)','Vehicle Category','Number of Vehicles Involved',(10,6),45),
      ('vehicle-ditribution','fig3_7_veh_type','Distribution of Vehicles Involved in Road Traffic Crashes (2021-2025)','Vehicle Type','Number of Vehicles Involved',(10,6),45)]
out={}
for k,f,t,xl,yl,sz,rot in spec:
    s=load_nbs(k).groupby('CAT').N.sum().sort_index(); out[f]=s.astype(int).to_dict()
    fig,ax=plt.subplots(figsize=sz); sns.barplot(x=s.index,y=s.values,hue=s.index,palette='viridis',legend=False,ax=ax)
    ax.set_title(t); ax.set_xlabel(xl); ax.set_ylabel(yl)
    for i,v in enumerate(s.values): ax.text(i,v,f'{int(v):,}',ha='center',va='bottom',fontsize=8)
    if rot: plt.setp(ax.get_xticklabels(),rotation=rot,ha='right')
    plt.tight_layout(); fig.savefig(os.path.join(OD,'figs',f'{f}.png'),dpi=150); plt.close()
json.dump(out,open(os.path.join(OD,'eda_totals.json'),'w'),indent=1)
for f,s in out.items(): print(f,sum(s.values()),{k:round(100*v/sum(s.values()),1) for k,v in s.items()})

fig3_4_killed_sex 28452 {'FEMALE ADULT': 15.7, 'FEMALE CHILD': 3.3, 'MALE ADULT': 77.7, 'MALE CHILD': 3.3}
fig3_5_injured_sex 173430 {'FEMALE ADULT': 20.9, 'FEMALE CHILD': 2.6, 'MALE ADULT': 73.5, 'MALE CHILD': 3.0}
fig3_6_veh_category 87955 {'COMMERCIAL': 68.2, 'DIPLOMAT': 0.0, 'GOVERNMENT': 1.2, 'PRIVATE': 30.5}
fig3_7_veh_type 87955 {'BICYCLE': 0.2, 'CAR': 28.0, 'LUXURY BUS': 0.3, 'MINIBUS': 18.3, 'MOTORCYCLE': 21.0, 'OTHERS': 1.5, 'PICK-UP': 2.7, 'SUV(JEEP)': 3.1, 'TANKER': 1.8, 'TRAILER': 5.3, 'TRICYCLE': 4.1, 'TRUCK': 13.2, 'VAN': 0.3}


# 5. Additional checks: classical time-series models, residual diagnostics, MCC and sensitivity of the severity target
These analyses use the predictions and tables already written by Section 2 (they do not refit the selected models), plus new ARIMA/ETS fits and a small set of random-forest refits for the target-definition sensitivity analysis.

In [15]:
import os, json, time, warnings, itertools
import numpy as np, pandas as pd
from scipy import stats
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import matthews_corrcoef, cohen_kappa_score, accuracy_score, f1_score, mean_absolute_error
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import het_breuschpagan, het_white, acorr_ljungbox
from statsmodels.stats.stattools import durbin_watson
import statsmodels.api as sm
warnings.filterwarnings('ignore')
OUT_DIR = os.environ.get('OUT_DIR', '.'); SEED=42; NB=5000
rng_idx = lambda n: np.random.default_rng(SEED).integers(0, n, (NB, n))
ci = lambda a: [float(np.percentile(a,2.5)), float(np.percentile(a,97.5))]
R5 = {}
raw = pd.read_csv(OUT_DIR+'/raw_full_2020Q4_2026Q1.csv').sort_values(['STATE','YEAR','QUARTER_NUM'])
raw['T'] = (raw.YEAR-2020)*4 + raw.QUARTER_NUM            # 2020-Q4 = 4 ... 2026-Q1 = 25
pr = pd.read_csv(OUT_DIR+'/pred_reg_2025.csv')
states = sorted(raw.STATE.unique())

## A. Classical time-series benchmarks (ARIMA and exponential smoothing), fitted state by state
One-step-ahead forecasts for each 2025 quarter, refitting on all earlier data (expanding window), so the information set matches the machine-learning models and the naive baseline.

In [16]:
def aicc(res, n):
    k = len(res.params); return res.aic + 2*k*(k+1)/max(n-k-1,1)
ARIMA_GRID = [(p,d,q) for p in (0,1,2) for d in (0,1) for q in (0,1)]
def fit_arima(y):
    best=None
    for o in ARIMA_GRID:
        try:
            r = ARIMA(y, order=o, trend='c' if o[1]==0 else 'n').fit()
            sc = aicc(r, len(y))
            if np.isfinite(sc) and (best is None or sc<best[0]): best=(sc,r,o)
        except Exception: pass
    return best
def fit_ets(y):
    best=None
    for kw in (dict(trend=None), dict(trend='add', damped_trend=True)):
        try:
            r = ExponentialSmoothing(y, initialization_method='estimated', **kw).fit()
            sc = r.aicc
            if np.isfinite(sc) and (best is None or sc<best[0]): best=(sc,r,kw)
        except Exception: pass
    return best
def one_step(series_by_state, targets):
    out=[]
    for s in states:
        d = series_by_state[s]
        for (yr,q) in targets:
            t=(yr-2020)*4+q; hist=d[d['T']<t]['TOTAL CASES'].values.astype(float)
            a=fit_arima(hist); e=fit_ets(hist)
            out.append(dict(STATE=s,YEAR=yr,QUARTER_NUM=q, ARIMA=float(max(a[1].forecast(1)[0],0)), ARIMA_order=str(a[2]),
                            ETS=float(max(e[1].forecast(1)[0],0)), ETS_form=('damped' if e[2].get('trend') else 'SES')))
    return pd.DataFrame(out)
ser = {s: raw[raw.STATE==s] for s in states}
t0=time.time(); ts25 = one_step(ser, [(2025,q) for q in (1,2,3,4)]); print('2025 TS fits %.0fs'%(time.time()-t0))
ts26 = one_step(ser, [(2026,1)])
ts25.to_csv(OUT_DIR+'/pred_ts_2025.csv', index=False); ts26.to_csv(OUT_DIR+'/pred_ts_2026Q1.csv', index=False)
m = pr.merge(ts25, on=['STATE','YEAR','QUARTER_NUM'])
assert len(m)==148
y = m['actual'].values.astype(float); bi = rng_idx(len(y))
def row(name, p):
    e = np.abs(y-p); b = e[bi].mean(1); ss=((y-p)**2).sum(); r2 = 1-ss/((y-y.mean())**2).sum()
    return dict(model=name, MAE=float(e.mean()), lo=ci(b)[0], hi=ci(b)[1], RMSE=float(np.sqrt(((y-p)**2).mean())), R2=float(r2))
cands = {'ARIMA (state-wise, AICc order)':m['ARIMA'].values, 'Exponential smoothing (SES / damped trend)':m['ETS'].values,
         'Naive persistence (LAG_1)':m['Naive persistence (LAG_1)'].values, 'Gradient Boosting (selected)':m['Gradient Boosting'].values}
tt = pd.DataFrame([row(k,v) for k,v in cands.items()]); print(tt.round(3).to_string()); tt.to_csv(OUT_DIR+'/tab_ts_models.csv', index=False)
pw=[]
for k in ['ARIMA (state-wise, AICc order)','Exponential smoothing (SES / damped trend)']:
    for ref in ['Naive persistence (LAG_1)','Gradient Boosting (selected)']:
        d = (np.abs(y-cands[k])-np.abs(y-cands[ref]))[bi].mean(1)
        pw.append(dict(model=k, vs=ref, diff=float((np.abs(y-cands[k])-np.abs(y-cands[ref])).mean()), lo=ci(d)[0], hi=ci(d)[1], P_better=float((d<0).mean())))
pw=pd.DataFrame(pw); print(pw.round(3).to_string()); pw.to_csv(OUT_DIR+'/tab_ts_pairwise.csv', index=False)
# prospective 2026-Q1
q26 = pd.read_csv(OUT_DIR+'/pred_reg_2026Q1_trained_2021_2025.csv')
act26 = raw[(raw.YEAR==2026)&(raw.QUARTER_NUM==1)].set_index('STATE')['TOTAL CASES']; mm26 = ts26.set_index('STATE')
a26 = act26.reindex(states).values.astype(float)
R5['ts_2026Q1'] = {k: float(np.abs(a26-mm26.reindex(states)[k].values).mean()) for k in ['ARIMA','ETS']}
R5['ts_orders'] = ts25['ARIMA_order'].value_counts().to_dict(); R5['ts_ets_forms']=ts25['ETS_form'].value_counts().to_dict()
# stationarity (ADF) on training series 2020-Q4..2024-Q4 (low power with 17 points)
adf=[]
for s in states:
    v = ser[s][ser[s]['T']<=20]['TOTAL CASES'].values.astype(float)
    try: adf.append(adfuller(v, maxlag=1, autolag=None)[1])
    except Exception: pass
R5['adf_share_reject_5pct'] = float(np.mean(np.array(adf)<0.05)); R5['adf_n']=len(adf)
print(R5)

2025 TS fits 51s


                                        model     MAE      lo      hi    RMSE     R2
0              ARIMA (state-wise, AICc order)  14.686  12.535  16.958  19.991  0.881
1  Exponential smoothing (SES / damped trend)  13.245  11.334  15.342  18.250  0.901
2                   Naive persistence (LAG_1)  14.757  12.676  17.068  19.982  0.881
3                Gradient Boosting (selected)  13.748  11.584  16.187  19.936  0.881
                                        model                            vs   diff     lo     hi  P_better
0              ARIMA (state-wise, AICc order)     Naive persistence (LAG_1) -0.070 -1.273  1.138     0.548
1              ARIMA (state-wise, AICc order)  Gradient Boosting (selected)  0.939 -0.910  2.719     0.156
2  Exponential smoothing (SES / damped trend)     Naive persistence (LAG_1) -1.511 -2.926 -0.099     0.981
3  Exponential smoothing (SES / damped trend)  Gradient Boosting (selected) -0.502 -2.154  1.073     0.724
{'ts_2026Q1': {'ARIMA': 12.9112275167020

## B. Regression residual diagnostics (2025 test residuals; OLS diagnostics for the linear model)

In [17]:
d = pr.copy(); d['t']=(d.YEAR-2025)*4+d.QUARTER_NUM
diag=[]
def lag1_pooled(df, col):
    df=df.sort_values(['STATE','t']); df['prev']=df.groupby('STATE')[col].shift(1); z=df.dropna(subset=['prev'])
    r,p = stats.pearsonr(z[col], z['prev']); return float(r), float(p), len(z)
for name,col in [('Gradient Boosting (selected)','Gradient Boosting'),('Naive persistence (LAG_1)','Naive persistence (LAG_1)'),('Linear Regression','Linear Regression')]:
    e = d['actual']-d[col]; d['e_'+col]=e; fit=d[col]
    r1,p1,n1 = lag1_pooled(d.copy(),'e_'+col)
    rho,prho = stats.spearmanr(np.abs(e), fit)                 # heteroskedasticity: error size vs level
    X = sm.add_constant(fit.values); bp = sm.OLS(e.values**2, X).fit()   # LM-type: e^2 on fitted
    lm = len(e)*bp.rsquared; plm = float(1-stats.chi2.cdf(lm,1))
    sw = stats.shapiro(e); tert = pd.qcut(d.groupby('STATE')['actual'].transform('mean'), 3, labels=['low','mid','high'])
    sds = e.groupby(tert).std().to_dict()
    diag.append(dict(model=name, mean_resid=float(e.mean()), sd=float(e.std()), lag1_r=r1, lag1_p=p1, lag1_n=n1, spearman_abs_vs_fitted=float(rho), spearman_p=float(prho),
                     bp_LM=float(lm), bp_p=plm, shapiro_W=float(sw.statistic), shapiro_p=float(sw.pvalue), skew=float(stats.skew(e)), kurt=float(stats.kurtosis(e)),
                     sd_low=float(sds['low']), sd_mid=float(sds['mid']), sd_high=float(sds['high'])))
dg = pd.DataFrame(diag); print(dg.round(4).T.to_string()); dg.to_csv(OUT_DIR+'/tab_reg_diagnostics_test.csv', index=False)
# in-sample OLS diagnostics for the linear model (training years), proper tests
reg = pd.read_csv(OUT_DIR+'/regression_features_2021_2025_v2.csv')
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder().fit(states); reg['STATE_ENC']=le.transform(reg['STATE'])
FR = ['YEAR','QUARTER_NUM','TIME_INDEX','STATE_ENC','LAG_1','LAG_2','LAG_4','ROLLING_MEAN_2','ROLLING_MEAN_4','TREND']
tr = reg[reg.YEAR<=2024].sort_values(['STATE','TIME_INDEX'])
ols = sm.OLS(tr['TOTAL CASES'].astype(float), sm.add_constant(tr[FR].astype(float))).fit()
bpt = het_breuschpagan(ols.resid, ols.model.exog); wh = het_white(ols.resid, ols.model.exog)
tr = tr.assign(res=ols.resid.values)
num=den=0.0
for s,g in tr.groupby('STATE'):
    e=g['res'].values; num+=((e[1:]-e[:-1])**2).sum(); den+=(e**2).sum()
dw_pooled = float(num/den)
natres = tr.groupby('TIME_INDEX')['res'].sum()
lb = acorr_ljungbox(natres, lags=[1,4], return_df=True)
R5['ols_train'] = dict(n=int(len(tr)), R2=float(ols.rsquared), BP_LM=float(bpt[0]), BP_p=float(bpt[1]), White_LM=float(wh[0]), White_p=float(wh[1]),
                       DW_pooled=dw_pooled, LB_lag1_p=float(lb['lb_pvalue'].iloc[0]), LB_lag4_p=float(lb['lb_pvalue'].iloc[1]), LB_n=int(len(natres)))
# log-scale check: do relative errors look homoskedastic?
sel = d['Gradient Boosting']; rel = (d['actual']-sel).abs()/d['actual'].clip(lower=1)
tert_all = pd.qcut(d.groupby('STATE')['actual'].transform('mean'), 3, labels=['low','mid','high']); sig=Rj0['reg_sigma_2025'] if False else None
R5['rel_err_median']=float(rel.median()); R5['rel_err_spearman_vs_level']=[float(x) for x in stats.spearmanr(rel, d['actual'])]
print(json.dumps(R5, indent=1))
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
os.makedirs(OUT_DIR+'/figs', exist_ok=True)
e = d['e_Gradient Boosting']; fit=d['Gradient Boosting']
fig, ax = plt.subplots(2,2, figsize=(9,7))
ax[0,0].scatter(fit, e, s=14, alpha=.7, color='#1f3b5c'); ax[0,0].axhline(0,color='k',lw=.8); ax[0,0].set_xlabel('Fitted (predicted) cases'); ax[0,0].set_ylabel('Residual (actual − predicted)'); ax[0,0].set_title('(a) Residuals versus fitted')
ax[0,1].scatter(fit, e.abs(), s=14, alpha=.7, color='#b5532c'); ax[0,1].set_xlabel('Fitted (predicted) cases'); ax[0,1].set_ylabel('Absolute residual'); ax[0,1].set_title('(b) Error size versus level')
stats.probplot(e, dist='norm', plot=ax[1,0]); ax[1,0].set_title('(c) Normal Q–Q plot'); ax[1,0].get_lines()[0].set_markersize(4)
dd = d.sort_values(['STATE','t']); dd['prev']=dd.groupby('STATE')['e_Gradient Boosting'].shift(1); z=dd.dropna(subset=['prev'])
ax[1,1].scatter(z['prev'], z['e_Gradient Boosting'], s=14, alpha=.7, color='#2a7f62'); ax[1,1].axhline(0,color='k',lw=.8); ax[1,1].axvline(0,color='k',lw=.8)
ax[1,1].set_xlabel('Previous-quarter residual (same state)'); ax[1,1].set_ylabel('Residual'); ax[1,1].set_title('(d) Residual autocorrelation (lag 1)')
for a in ax.ravel(): a.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.savefig(OUT_DIR+'/figs/fig4_10_residual_diagnostics.png', dpi=150); plt.close()

                                                   0                          1                  2
model                   Gradient Boosting (selected)  Naive persistence (LAG_1)  Linear Regression
mean_resid                                    2.2518                     1.1622            10.8433
sd                                           19.8758                    20.0157            18.6457
lag1_r                                       -0.1489                    -0.4221            -0.1399
lag1_p                                        0.1188                        0.0             0.1431
lag1_n                                           111                        111                111
spearman_abs_vs_fitted                        0.5244                     0.4789             0.3418
spearman_p                                       0.0                        0.0                0.0
bp_LM                                         34.808                    35.9678            26.8691
bp_p      

## C. Matthews correlation coefficient (MCC) for all classifiers and baselines

In [18]:
pc = pd.read_csv(OUT_DIR+'/pred_cls_2025.csv'); yc = pc['actual'].values; bj = rng_idx(len(yc))
mcc=[]
for col in pc.columns[4:]:
    p=pc[col].values; v=matthews_corrcoef(yc,p); b=np.array([matthews_corrcoef(yc[ix],p[ix]) for ix in bj[:2000]])
    mcc.append(dict(model=col, MCC=float(v), lo=ci(b)[0], hi=ci(b)[1]))
mcc=pd.DataFrame(mcc); print(mcc.round(3).to_string()); mcc.to_csv(OUT_DIR+'/tab_cls_mcc.csv', index=False)

                                model    MCC     lo     hi
0                 Logistic Regression  0.141  0.025  0.259
1                       Random Forest  0.163  0.050  0.282
2                   Gradient Boosting  0.190  0.069  0.310
3               Random Forest (tuned)  0.278  0.160  0.394
4           Gradient Boosting (tuned)  0.258  0.141  0.379
5                      Majority class  0.000  0.000  0.000
6  State modal class (training years)  0.358  0.236  0.474
7              Previous-quarter class  0.300  0.179  0.419


## D. Sensitivity of the conclusions to the definition of the severity target
The reference definition (tertiles of the fatality rate) is arbitrary. The same random-forest configuration is re-estimated under alternative definitions, and compared with the same two reference rules (majority class, each state's usual class in the training years).

In [19]:
cls = pd.read_csv(OUT_DIR+'/classification_features_2021_2025_v2.csv'); cls['STATE_ENC']=le.transform(cls['STATE'])
FACT=['SPV','UPD','TBT','MDV','BFL','OVL','DOT','WOT','DGD','BRD','RTV','OBS','SOS','DAD','PWR','FTQ','SLV','OTHERS']; PROPS=[f+'_PROP' for f in FACT]
FC=['YEAR','QUARTER_NUM','TIME_INDEX','STATE_ENC']+PROPS
Rj = json.load(open(OUT_DIR+'/results.json')); prm = Rj['cls_tuned_params']['Random Forest (tuned)']
prm = {k:v for k,v in prm.items() if v is not None}
trn = cls[cls.YEAR<=2024]; tst = cls[cls.YEAR==2025]
fr = trn['FATALITY_RATE']
defs = {'Tertiles of the training fatality rate (reference)': lambda x: np.digitize(x, [fr.quantile(1/3), fr.quantile(2/3)], right=True),
        'Fixed cut-points 0.25 and 0.50 deaths per crash': lambda x: np.digitize(x, [0.25, 0.50], right=True),
        'Quartile-based extremes (bottom, middle half, top)': lambda x: np.digitize(x, [fr.quantile(.25), fr.quantile(.75)], right=True),
        'Binary: above versus at or below the training median': lambda x: (x > fr.median()).astype(int)}
sens=[]
for nm,f in defs.items():
    ytr=f(trn['FATALITY_RATE'].values); yte=f(tst['FATALITY_RATE'].values)
    if nm.startswith('Tertiles'): ytr=trn['SEVERITY'].map({'Low':0,'Medium':1,'High':2}).values; yte=tst['SEVERITY'].map({'Low':0,'Medium':1,'High':2}).values
    rf=RandomForestClassifier(random_state=SEED, n_jobs=-1, **prm).fit(trn[FC], ytr); p=rf.predict(tst[FC])
    maj=np.full(len(yte), pd.Series(ytr).mode()[0]); sm_mode=pd.Series(ytr,index=trn.index).groupby(trn['STATE']).agg(lambda s:s.mode().iloc[0]); pm=tst['STATE'].map(sm_mode).values.astype(int)
    ds = ((p==yte).astype(float)-(pm==yte).astype(float)); b=rng_idx(len(yte)); dm=((p==yte).astype(float)-(maj==yte).astype(float))
    sens.append(dict(definition=nm, classes=len(np.unique(ytr)), rf_acc=float((p==yte).mean()), maj_acc=float((maj==yte).mean()), modal_acc=float((pm==yte).mean()),
                     rf_minus_maj=float(dm.mean()), maj_lo=ci(dm[b].mean(1))[0], maj_hi=ci(dm[b].mean(1))[1],
                     rf_minus_modal=float(ds.mean()), modal_lo=ci(ds[b].mean(1))[0], modal_hi=ci(ds[b].mean(1))[1],
                     kappa=float(cohen_kappa_score(yte,p)), mcc=float(matthews_corrcoef(yte,p))))
sens=pd.DataFrame(sens); print(sens.round(3).T.to_string()); sens.to_csv(OUT_DIR+'/tab_cls_target_sensitivity.csv', index=False)
# continuous fatality-rate regression: RF versus state mean and previous-quarter rate (MAE in deaths per crash)
cs = cls.sort_values(['STATE','TIME_INDEX']).copy(); cs['PREV_RATE']=cs.groupby('STATE')['FATALITY_RATE'].shift(1)
trc=cs[cs.YEAR<=2024].dropna(subset=['PREV_RATE']); tec=cs[cs.YEAR==2025]
rfr=RandomForestRegressor(n_estimators=300,max_depth=5,min_samples_split=10,max_features=0.5,random_state=SEED,n_jobs=-1).fit(trn[FC], trn['FATALITY_RATE'])
prf=rfr.predict(tst[FC]); smean=tst['STATE'].map(trn.groupby('STATE')['FATALITY_RATE'].mean()).values; pprev=tec.loc[tst.index,'PREV_RATE'].values
yv=tst['FATALITY_RATE'].values; bb=rng_idx(len(yv))
def mae_row(n,p): 
    e=np.abs(yv-p); return dict(model=n, MAE=float(e.mean()), lo=ci(e[bb].mean(1))[0], hi=ci(e[bb].mean(1))[1])
cr=pd.DataFrame([mae_row('Random Forest (fatality rate)',prf), mae_row('State mean rate (training years)',smean), mae_row('Previous-quarter rate',pprev), mae_row('Overall training mean',np.full(len(yv),fr.mean()))])
dd1=(np.abs(yv-prf)-np.abs(yv-smean)); R5['rate_rf_minus_statemean']=[float(dd1.mean())]+ci(dd1[bb].mean(1))
print(cr.round(4)); cr.to_csv(OUT_DIR+'/tab_rate_regression.csv', index=False)
json.dump(R5, open(OUT_DIR+'/results_extra.json','w'), indent=1)

# coverage of the heuristic one-step band (±1.96 sigma) by volume tercile
sigma = json.load(open(OUT_DIR+'/results.json'))['reg_sigma_2025']; sigma = sigma if np.isscalar(sigma) else float(list(sigma.values())[0])
inside = (np.abs(d['actual']-d['Gradient Boosting'])<=1.96*sigma)
R5['band_sigma']=float(sigma); R5['band_coverage_by_tercile']={k:float(v) for k,v in inside.groupby(tert_all).mean().items()}; R5['band_coverage_all']=float(inside.mean())
print(R5['band_coverage_by_tercile'], R5['band_coverage_all'])
json.dump(R5, open(OUT_DIR+'/results_extra.json','w'), indent=1)

                                                                 0                                                1                                                   2                                                     3
definition      Tertiles of the training fatality rate (reference)  Fixed cut-points 0.25 and 0.50 deaths per crash  Quartile-based extremes (bottom, middle half, top)  Binary: above versus at or below the training median
classes                                                          3                                                3                                                   3                                                     2
rf_acc                                                       0.507                                            0.547                                               0.446                                                 0.601
maj_acc                                                      0.351                                            0.

                              model     MAE      lo      hi
0     Random Forest (fatality rate)  0.2531  0.2208  0.2863
1  State mean rate (training years)  0.2056  0.1756  0.2374
2             Previous-quarter rate  0.2578  0.2163  0.3041
3             Overall training mean  0.2682  0.2344  0.3024
{'low': 1.0, 'mid': 1.0, 'high': 0.8125} 0.9391891891891891
